In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ClinicalDistill — Qwen 1.5 1.8B QLoRA (Kaggle P100)
## Experiment 6

Same model as Experiment 5 (Qwen1.5-1.8B) but with 4-bit quantization.
Lets us compare LoRA vs QLoRA on the same model — same tradeoff analysis as Gemma.

| Model             | Params | Valid JSON | F1    | Urgent Acc |
|-------------------|--------|------------|-------|------------|
| Gemma-3-1B LoRA   | 1B     | 100%       | 0.781 | 85.7%      |
| Gemma-3-1B QLoRA  | 1B     | 100%       | 0.740 | 82.9%      |
| Qwen1.5-1.8B LoRA | 1.8B   | ___        | ___   | ___        |
| Qwen1.5-1.8B QLoRA| 1.8B   | ___        | ___   | ___        |

**Before running:** Settings → Internet → On

In [2]:
!pip install -q bitsandbytes transformers peft accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.8 MB/s eta 0:00:00


In [3]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
VRAM: 15.6 GB


In [5]:
import os

PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

print(os.listdir(PROJECT_PATH))
!wc -l {PROJECT_PATH}/data/train_fixed.jsonl {PROJECT_PATH}/data/test_fixed.jsonl

['README.md', 'requirements.txt', '.env', '.gitignore', 'data', 'schema', '.git', 'models']
  145 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train_fixed.jsonl
   35 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test_fixed.jsonl
  180 total


In [6]:
from datasets import load_dataset
import json

# Canonical dataset — same across all experiments for fair comparison
# 145 train: 120 formal clinical (GPT-4o) + 25 casual/colloquial (Claude)
#  35 test:  30 formal + 5 casual
dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test":  f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print("\nSample example:")
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 145
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 35
    })
})

Sample example:
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours', 'unspecified'], 'severity': ['severe', 'unspecified'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


In [7]:
from huggingface_hub import login
import json
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)
    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.</input>
<output>{"symptoms": ["chest pain", "shortness of breath"], "duration": ["2 hours", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen1.5-1.8B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model: {model_id}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Model: Qwen/Qwen1.5-1.8B-Chat
Parameters: 1,836,828,672
VRAM used: 1.89 GB


In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 1,839,974,400 || trainable%: 0.1710


In [16]:
from trl import SFTTrainer, SFTConfig
import time

args = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/qwen-qlora",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

start_time = time.time()
print("Starting Qwen1.5 QLoRA training...")
trainer.train()
training_time = time.time() - start_time

print(f"Training complete: {training_time:.0f}s ({training_time/60:.1f} min)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Starting Qwen1.5 QLoRA training...


Step,Training Loss
10,0.277790
20,0.255372
30,0.220300
40,0.204049
50,0.187565
60,0.167568
70,0.160762
80,0.161853
90,0.139501
100,0.129092


Training complete: 1313s (21.9 min)
VRAM: 5.07 GB


In [17]:
SAVE_PATH = f"{OUTPUT_DIR}/qwen-qlora-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to: {SAVE_PATH}")

Saved to: /kaggle/working/models/qwen-qlora-final


In [18]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

SAVE_PATH = f"{OUTPUT_DIR}/qwen-qlora-final"

tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)
tokenizer_loaded.pad_token = tokenizer_loaded.eos_token
tokenizer_loaded.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen1.5-1.8B-Chat",
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer_loaded.pad_token_id
model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)

print("Model loaded for inference")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded for inference
VRAM: 5.71 GB


In [19]:
def extract_clinical(text):
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.eos_token_id
        )
    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()


test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]
for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

INPUT: Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.
OUTPUT: {"symptoms": ["headache", "blurred vision", "confusion"], "duration": ["2 days", "unspecified", "unspecified"], "severity": ["severe", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Child presents with mild fever since yesterday, runny nose and occasional cough.
OUTPUT: {"symptoms": ["fever", "nose", "cough"], "duration": ["yesterday", "unspecified", "unspecified"], "severity": ["mild", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Patient reports sharp lower back pain for a week, worse when sitting, no fever.
OUTPUT: {"symptoms": ["sharp back pain"], "duration": ["1 week"], "severity": ["weak"], "urgent": false}</output>
------------------------------------------------------------


In [20]:
import json

def parse_output(text):
    try:
        text = text.strip().replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))
    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

valid_json_count = 0
f1_scores = []
urgent_correct = 0

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]
    if is_valid:
        valid_json_count += 1
        f1_scores.append(symptom_f1(pred.get("symptoms", []), true.get("symptoms", [])))
        if pred.get("urgent") == true.get("urgent"):
            urgent_correct += 1

total = len(dataset["test"])
avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Qwen1.5-1.8B QLoRA")
print("=" * 50)
print(f"Valid JSON rate : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1  : {avg_f1:.3f}")
print(f"Urgent Accuracy : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

EVALUATION RESULTS — Qwen1.5-1.8B QLoRA
Valid JSON rate : 33/35 (94.3%)
Avg Symptom F1  : 0.696
Urgent Accuracy : 29/33 (87.9%)
